In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
import win32com.client as win32
import time  # Для измерения времени выполнения
import shutil
import re
from glob import glob
import gc
from datetime import timedelta

# Функция для форматирования времени в часы, минуты и секунды
def format_elapsed_time(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{int(hours)} часа(ов) {int(minutes)} минут(ы) {seconds:.2f} секунд"

# Функция для проверки, является ли файл скрытым (для Windows)
def is_hidden(file_path):
    try:
        # Получаем атрибуты файла
        file_attributes = os.stat(file_path).st_file_attributes
        # Проверяем, установлен ли флаг "скрытый"
        return file_attributes & 2 != 0  # 2 соответствует атрибуту "скрытый"
    except Exception:
        # Если возникла ошибка, считаем файл не скрытым
        return False

# Функция для форматирования даты в строковый формат 'YYYY-MM-DD'
def format_date_column(df, date_column):
    if date_column in df.columns:
        df[date_column] = pd.to_datetime(df[date_column], errors='coerce').dt.strftime('%Y-%m-%d')
    return df

# Функция для подключения к SQL Server с аутентификацией Windows
def connect_to_sql(server, database):
    connection_string = (
        f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server"
        "&trusted_connection=yes"
    )
    engine = create_engine(connection_string)
    return engine

# Функция для обработки ошибок и замены их на null
def handle_errors(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].apply(lambda x: None if isinstance(x, str) and x.strip() == '' else x)
    return df

In [2]:
FOLDER_PATH = os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!")
FOLDER_PATH_FEATURES = r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Дашбоард по рекламным кампаниям"
FOLDER_PATH_FOR_DB= os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям")
FOLDER_PATH_DUDL = os.path.normpath(r"\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ")

SQL_SERVER = "cl01sql"
SQL_DATABASE_DBREPORT = "DBReport"
SQL_DATABASE_DBPARTNERS = "DBPartners"

In [3]:
print("Начинаем собирать Базу Данных...")
start_all_time = time.time()
engine = connect_to_sql(SQL_SERVER, SQL_DATABASE_DBPARTNERS)

Начинаем собирать Базу Данных...


In [4]:
# 4. Получить данные из файла "Справочник.xlsx"
try:
    print("Начинаем получать данные для Справочника...")
    start_time = time.time()  # Запускаем таймер
    file_path_reference = os.path.join(FOLDER_PATH, "Справочник.xlsx")

    if os.path.exists(file_path_reference):
        # Список столбцов, которые нужно взять из файла
        columns_to_read = [
            "Артикул", "Артикул OZ", "Наименование", "Коллекция",
            "Бренд", "Размер", "Сезон", "Направление", "Розничный отдел",
            "Модель", "Группа", "Бизнес-группа", "Техсегмент",
            "Байер", "Две последние коллекции", "Основной артикул", "Ответственный за группу", "Себестоимость с НДС",
            "Процент выкупа", "НДС", "Группа для отчетов"
        ]

        # Типы данных для столбцов
        column_dtypes = {
            "Артикул": str,
            "Артикул OZ": str,
            "Наименование": str,
            "Коллекция": str,
            "Размер": str,
            "Бренд": str,
            "Сезон": str,
            "Направление": str,
            "Розничный отдел": str,
            "Модель": str,
            "Группа": str,
            "Бизнес-группа": str,
            "Техсегмент": str,
            "Байер": str,
            "Две последние коллекции": str,
            "Основной артикул": str,
            "Ответственный за группу": str,
            "Себестоимость с НДС": float,
            "Процент выкупа": float,
            "НДС": int,
            "Группа для отчетов": str
        }

        # Чтение файла с указанием нужных столбцов и типов данных
        df_reference = pd.read_excel(
            file_path_reference,
            sheet_name="Выгрузка для справочника",
            engine="openpyxl",
            usecols=columns_to_read,
            dtype=column_dtypes
        )

        # Удаление дубликатов
        df_reference = df_reference.drop_duplicates()

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Справочник:")
        print(df_reference.head())

        # Сохраняем результат
        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Справочника успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Справочник.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Справочника: {e}")

Начинаем получать данные для Справочника...
Первые 5 строк таблицы Справочник:
    Артикул                         Наименование Размер Коллекция Бренд Сезон  \
0  W9009967  Полуботинки женские зимние ZL25AW-5     38    2025AW  kari  зима   
1  W9009967  Полуботинки женские зимние ZL25AW-5     36    2025AW  kari  зима   
2  W9009967  Полуботинки женские зимние ZL25AW-5     40    2025AW  kari  зима   
3  W9009967  Полуботинки женские зимние ZL25AW-5     37    2025AW  kari  зима   
4  W9009967  Полуботинки женские зимние ZL25AW-5     41    2025AW  kari  зима   

     Направление Розничный отдел    Модель Бизнес-группа  ... Техсегмент  \
0  Женская обувь   Женская обувь  ZL25AW-5         Обувь  ...   flat (L)   
1  Женская обувь   Женская обувь  ZL25AW-5         Обувь  ...   flat (L)   
2  Женская обувь   Женская обувь  ZL25AW-5         Обувь  ...   flat (L)   
3  Женская обувь   Женская обувь  ZL25AW-5         Обувь  ...   flat (L)   
4  Женская обувь   Женская обувь  ZL25AW-5         Обу

In [5]:
# 7. Создание таблицы "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу ВсегоРазмеров...")
    start_time = time.time()  # Запускаем таймер

    # Проверка наличия необходимых столбцов
    required_columns = ["Артикул", "Размер"]
    for col in required_columns:
        if col not in df_reference.columns:
            print(f"Ошибка: Отсутствует столбец '{col}' в df_reference.")
            exit()

    # Очищаем столбец "Размер":
    # - Преобразуем в строковый формат
    # - Удаляем лишние пробелы
    # - Заменяем пустые строки на None
    df_reference["Размер"] = df_reference["Размер"].astype(str).str.strip().replace('', None)

    # Создаем DataFrame с количеством размеров для каждого артикула
    df_reference_unique = (
        df_reference
        .drop_duplicates(subset=["Артикул", "Размер"])  # Удаляем дубликаты Артикул-Размер
        .groupby("Артикул")["Размер"]  # Группируем по артикулу
        .apply(lambda sizes: len(sizes.dropna().unique()) if len(sizes.dropna()) > 0 else 1)  # Подсчитываем размеры
        .reset_index(name="Всего размеров")
    )

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ВсегоРазмеров:")
    print(df_reference_unique.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ВсегоРазмеров успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ВсегоРазмеров: {e}")

Начинаем создавать таблицу ВсегоРазмеров...
Первые 5 строк таблицы ВсегоРазмеров:
    Артикул  Всего размеров
0  00001851               5
1  00001852               5
2  00001855               5
3  00001856               5
4  00001931               6
Таблица ВсегоРазмеров успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 32.35 секунд


In [6]:
# 6. Получить данные таблицы с SQL (РазмерыНаАгрегаторе)
try:
    print("Начинаем получать данные для РазмеровНаАгрегаторе...")
    start_time = time.time()  # Запускаем таймер
    query_sizes = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], COUNT(DISTINCT(a.[INVENTSIZEID])) AS [Колво размеров]
        FROM [DBPartners].[dbo].[WblmRepGetStockOzon] a
        WHERE [dt] >= '{(pd.Timestamp.today() - pd.DateOffset(months=3)).strftime("%Y-%m-%d")}'
        GROUP BY [dt], [itemid]
    """
    df_sizes = pd.read_sql(query_sizes, engine)
    df_sizes = format_date_column(df_sizes, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы РазмерыНаАгрегаторе:")
    print(df_sizes.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для РазмеровНаАгрегаторе: {e}")

Начинаем получать данные для РазмеровНаАгрегаторе...
Первые 5 строк таблицы РазмерыНаАгрегаторе:
         Дата   Артикул  Колво размеров
0  2025-09-02  00006170               3
1  2025-09-02  00006410               2
2  2025-09-02  00008060               1
3  2025-09-02  00128845               4
4  2025-09-02  00146325               6
Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: 0 часа(ов) 0 минут(ы) 53.41 секунд


In [7]:
# 8. Связать "РазмерыНаАгрегаторе" с "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу Дистрибуция...")
    start_time = time.time()  # Запускаем таймер

    # Объединяем таблицы по полю "Артикул"
    df_distribution = pd.merge(df_sizes, df_reference_unique, on="Артикул", how="left")

    # Вычисляем дистрибуцию с проверкой на деление на ноль
    df_distribution["Дистрибуция"] = df_distribution.apply(
        lambda row: row["Колво размеров"] / row["Всего размеров"] if row["Всего размеров"] != 0 else 0,
        axis=1
    )

    # Оставляем только нужные столбцы
    df_distribution = df_distribution[["Дата", "Артикул", "Дистрибуция"]]

    # Форматирование даты
    df_distribution = format_date_column(df_distribution, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Дистрибуция:")
    print(df_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Дистрибуция успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Дистрибуция: {e}")

Начинаем создавать таблицу Дистрибуция...
Первые 5 строк таблицы Дистрибуция:
         Дата   Артикул  Дистрибуция
0  2025-09-02  00006170     0.500000
1  2025-09-02  00006410     0.333333
2  2025-09-02  00008060     0.166667
3  2025-09-02  00128845     1.000000
4  2025-09-02  00146325     1.000000
Таблица Дистрибуция успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 32.73 секунд


In [8]:
# 2. Получить данные таблицы с SQL (Остатки)
try:
    print("Начинаем получать данные для Остатков...")
    start_time = time.time()  # Запускаем таймер
    query_stock = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], SUM(a.[free_to_sell_amount]) AS [Остаток Агрегатора]
        FROM [DBPartners].[dbo].[WblmRepGetStockOzon] a
        WHERE [dt] >= '{(pd.Timestamp.today() - pd.DateOffset(months=3)).strftime("%Y-%m-%d")}'
        GROUP BY [dt], [itemid]
    """
    df_stock = pd.read_sql(query_stock, engine)
    df_stock = format_date_column(df_stock, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатков:")
    print(df_stock.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для Остатков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для Остатков: {e}")

Начинаем получать данные для Остатков...
Первые 5 строк таблицы Остатков:
         Дата   Артикул  Остаток Агрегатора
0  2025-09-02  00006170                   3
1  2025-09-02  00006410                   0
2  2025-09-02  00008060                   1
3  2025-09-02  00128845                 173
4  2025-09-02  00146325                 190
Данные для Остатков успешно сохранены. Время выполнения: 0 часа(ов) 0 минут(ы) 40.00 секунд


In [9]:
# 9. Связать "Остатки" с "Дистрибуция"
try:
    print("Начинаем создавать таблицу Остатки с дистрибуцией...")
    start_time = time.time()  # Запускаем таймер
    df_stock_with_distribution = pd.merge(df_stock, df_distribution, on=["Дата", "Артикул"], how="left")
    df_stock_with_distribution = format_date_column(df_stock_with_distribution, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатки с дистрибуцией:")
    print(df_stock_with_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Остатки с дистрибуцией успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Остатки с дистрибуцией: {e}")

Начинаем создавать таблицу Остатки с дистрибуцией...
Первые 5 строк таблицы Остатки с дистрибуцией:
         Дата   Артикул  Остаток Агрегатора  Дистрибуция
0  2025-09-02  00006170                   3     0.500000
1  2025-09-02  00006410                   0     0.333333
2  2025-09-02  00008060                   1     0.166667
3  2025-09-02  00128845                 173     1.000000
4  2025-09-02  00146325                 190     1.000000
Таблица Остатки с дистрибуцией успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 3.95 секунд


In [10]:
del df_stock

In [11]:
# 3. Получить данные из файлов вложенной папки "Показатели по дням"
try:
    print("Начинаем получать данные для Воронки...")
    start_time = time.time()  # Запускаем таймер
    folder_path_weeks = os.path.join(FOLDER_PATH, "Показатели по дням")
    df_funnel = pd.DataFrame()

    if os.path.exists(folder_path_weeks):
        for file in os.listdir(folder_path_weeks):
            file_path = os.path.join(folder_path_weeks, file)

            # Пропускаем скрытые файлы
            if is_hidden(file_path):
                print(f"Пропущен скрытый файл: {file}")
                continue

            # Проверяем расширение файла
            if file.endswith((".xlsx", ".xls")):
                try:
                    # Список столбцов, которые нужно взять из файла
                    columns_to_read = [
                        "Дата",	"Артикул", "Показы, всего", "Показы на карточке товара", "Показы в поиске и каталоге",
                        "Позиция в поиске и каталоге", "В корзину, всего", "Заказано товаров", "Отменено товаров",
                        "Доставлено товаров", "Возвращено товаров", "Заказано на сумму", "В корзину из карточки товара",
                        "Выкупили ШТ", "ТипАктивности", "Расход, ₽", "Продажи, ₽", "Заказы, шт", "Показы", "Клики",  "Цена"
                    ]

                    if file.endswith(".xlsx"):
                        temp_df = pd.read_excel(file_path, sheet_name="Воронка", engine="calamine", usecols=columns_to_read)
                    elif file.endswith(".xls"):
                        temp_df = pd.read_excel(file_path, sheet_name="Воронка", engine="xlrd", usecols=columns_to_read)

                    # Переименование столбцов
                    temp_df.rename(columns={
                        "Артикул": "Артикул",
                        "Продажи, ₽": "Рекламные заказано на сумму",
                        "Показы": "Рекламные показы",
                        "Клики": "Рекламные показы на карточке товара",
                        "Заказы, шт": "Рекламные заказано товаров"
                    }, inplace=True, errors="ignore")

                    # Типы данных для столбцов
                    column_dtypes = {
                        "Артикул": str,
                        "Показы, всего": int,
                        "Показы на карточке товара": int,
                        "Показы в поиске и каталоге": int,
                        "Позиция в поиске и каталоге": float,
                        "В корзину, всего": int,
                        "Заказано товаров": int,
                        "Отменено товаров": int,
                        "Доставлено товаров": int,
                        "Возвращено товаров": int,
                        "Заказано на сумму": int,
                        "Выкупили ШТ": int,
                        "В корзину из карточки товара": int,
                        "ТипАктивности": str,
                        "Рекламные заказано на сумму": int,
                        "Рекламные заказано товаров": int,
                        "Рекламные показы на карточке товара": int,
                        "Рекламные показы": int,
                        "Расход, ₽": int,
                        "Цена": int
                    }

                    # Форматирование даты
                    temp_df = format_date_column(temp_df, 'Дата')

                    df_funnel = pd.concat([df_funnel, temp_df])
                except Exception as e:
                    print(f"Ошибка при чтении файла {file}: {e}")

        # Удаление лишних столбцов (если они остались)
        df_funnel = df_funnel[[
             "Дата",	"Артикул", "ТипАктивности", "Показы, всего", "Показы на карточке товара", "Показы в поиске и каталоге",
                        "Позиция в поиске и каталоге", "В корзину, всего", "Заказано товаров", "Отменено товаров",
                        "Доставлено товаров", "Возвращено товаров", "Заказано на сумму", "В корзину из карточки товара",
                        "Выкупили ШТ", "Расход, ₽", "Рекламные заказано на сумму", "Рекламные заказано товаров",
                        "Рекламные показы", "Рекламные показы на карточке товара", "Цена"
        ]]
        mask = pd.to_numeric(df_funnel['Расход, ₽'], errors='coerce').eq(0)
        df_funnel.loc[mask, 'ТипАктивности'] = 'Органика'

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Воронка:")
        print(df_funnel.head())

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Воронки успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Папка 'Показатели по дням' не найдена.")
except Exception as e:
    print(f"Ошибка при получении данных для Воронки: {e}")

Начинаем получать данные для Воронки...
Первые 5 строк таблицы Воронка:
         Дата   Артикул ТипАктивности  Показы, всего  \
0  2025-10-01  31306230      Органика             34   
1  2025-10-01  17206150      Органика             64   
2  2025-10-01  17206200      Органика             14   
3  2025-10-01  17106050      Органика             40   
4  2025-10-01  17206190      Органика             80   

   Показы на карточке товара  Показы в поиске и каталоге  \
0                          8                          12   
1                          4                          48   
2                          0                           0   
3                          3                          18   
4                          7                          46   

   Позиция в поиске и каталоге  В корзину, всего  Заказано товаров  \
0                       192.83                 0                 0   
1                       234.82                 0                 0   
2                   

In [12]:
df_funnel

,Дата,Артикул,ТипАктивности,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,...,Возвращено товаров,Заказано на сумму,В корзину из карточки товара,Выкупили ШТ,"Расход, ₽",Рекламные заказано на сумму,Рекламные заказано товаров,Рекламные показы,Рекламные показы на карточке товара,Цена
0,2025-10-01,31306230,Органика,34,8,12,192.83,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,976.0
1,2025-10-01,17206150,Органика,64,4,48,234.82,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,621.0
2,2025-10-01,17206200,Органика,14,0,0,NaN,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,737.0
3,2025-10-01,17106050,Органика,40,3,18,142.89,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,618.0
4,2025-10-01,17206190,Органика,80,7,46,240.15,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,594.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49495,2025-10-31,c0101130,Органика,1,0,0,NaN,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
49496,2025-10-31,c1101020,Органика,3,1,1,NaN,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
49497,2025-10-31,u4906230,Органика,1,0,0,NaN,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
49498,2025-10-31,NaN,Органика,1,0,0,NaN,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
funnel_columns = df_funnel.columns

In [14]:
# Путь до папки
folder_path_weeks = os.path.join(FOLDER_PATH, "Затраты", "Озон. Затраты из Аналитики New Format")

# Собираем все .xlsx файлы
files = glob(os.path.join(folder_path_weeks, "*.xlsx"))

df_list = []
df_list_union = []
for file in files:
    # --- достаём дату из названия файла ---
    filename = os.path.basename(file)  # например: "Аналитика продвижения_16.09.2025.xlsx"
    date_str = filename.split("_")[-1].replace(".xlsx", "")  # "16.09.2025"
    date_parsed = (pd.to_datetime(date_str, format="%d.%m.%Y") - timedelta(days=1)).strftime("%Y-%m-%d")

    # читаем, пропуская первую строку
    df_tmp = pd.read_excel(file, engine='calamine', skiprows=1)
    df_tmp_union = pd.read_excel(file, sheet_name='Union', engine='calamine', skiprows=1)

    # оставляем только нужные колонки
    cols_keep = ["SKU", "ID кампании", "Инструмент", "Место размещения"]
    cols_keep_union = ["SKU в продвижении", "SKU из объединенной карточки", "Продажи, ₽", "Заказы, шт"]
    df_tmp = df_tmp[cols_keep]
    df_tmp_union = df_tmp_union[cols_keep_union]

    # добавляем колонку "Дата"
    df_tmp["Дата"] = date_parsed
    df_tmp_union["Дата"] = date_parsed

    df_list.append(df_tmp)
    df_list_union.append(df_tmp_union)

# объединяем все файлы
df_all = pd.concat(df_list, ignore_index=True)
df_all_union = pd.concat(df_list_union, ignore_index=True)
df_all.rename(columns={'SKU': "Артикул OZ"},inplace=True)
df_all_union.rename(columns={'SKU в продвижении': "Артикул OZ"},inplace=True)
df_all["Артикул OZ"] = df_all["Артикул OZ"].astype(str)
df_all_union["Артикул OZ"] = df_all_union["Артикул OZ"].astype(str)

In [15]:
df_all_union

,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт",Дата
0,1607739332,823892917,301.0,1,2025-09-30
1,2438516299,2310030768,2508.0,1,2025-09-30
2,1627853566,1627855021,755.0,1,2025-09-30
3,1627854626,1627852674,722.0,1,2025-09-30
4,1627857305,1829020570,2084.0,1,2025-09-30
...,...,...,...,...,...
672296,1267725462,351952384,5423.0,1,2025-10-30
672297,2659157998,2659158129,3068.0,1,2025-10-30
672298,2810541646,2810540473,496.0,1,2025-10-30
672299,1830569193,1830568337,1908.0,1,2025-10-30


In [16]:
df_reference

,Артикул,Наименование,Размер,Коллекция,Бренд,Сезон,Направление,Розничный отдел,Модель,Бизнес-группа,...,Техсегмент,Байер,Две последние коллекции,Основной артикул,Артикул OZ,Группа для отчетов,Себестоимость с НДС,НДС,Процент выкупа,Ответственный за группу
0,W9009967,Полуботинки женские зимние ZL25AW-5,38,2025AW,kari,зима,Женская обувь,Женская обувь,ZL25AW-5,Обувь,...,flat (L),Коновалова А.,2025AW,W9009967,2445277397,Обувь,1339.2297,20,0.860759,Гусева Дарья
1,W9009967,Полуботинки женские зимние ZL25AW-5,36,2025AW,kari,зима,Женская обувь,Женская обувь,ZL25AW-5,Обувь,...,flat (L),Коновалова А.,2025AW,W9009967,2445275855,Обувь,1339.2297,20,0.860759,Гусева Дарья
2,W9009967,Полуботинки женские зимние ZL25AW-5,40,2025AW,kari,зима,Женская обувь,Женская обувь,ZL25AW-5,Обувь,...,flat (L),Коновалова А.,2025AW,W9009967,2445275863,Обувь,1339.2297,20,0.860759,Гусева Дарья
3,W9009967,Полуботинки женские зимние ZL25AW-5,37,2025AW,kari,зима,Женская обувь,Женская обувь,ZL25AW-5,Обувь,...,flat (L),Коновалова А.,2025AW,W9009967,2445276684,Обувь,1339.2297,20,0.860759,Гусева Дарья
4,W9009967,Полуботинки женские зимние ZL25AW-5,41,2025AW,kari,зима,Женская обувь,Женская обувь,ZL25AW-5,Обувь,...,flat (L),Коновалова А.,2025AW,W9009967,2445276596,Обувь,1339.2297,20,0.860759,Гусева Дарья
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
702876,ya102020,Корзина для велосипеда черно-белая K7379,nan,2023SS,KariKids,лето,Летний спорт,Велосипеды,K7379,Принадлежности для спорта и активного отдыха,...,NaN,Бордачук Е. Спорт,"2022SS,2023SS",ya102020,NaN,NaN,55.1370,20,0.934197,Шляпин Алексей
702877,ya106020,Корзина для велосипеда черная K10997,nan,2024SS,Kari KIDS,лето,Летний спорт,Велосипеды,K10997,Принадлежности для спорта и активного отдыха,...,NaN,Бордачук Е. Спорт,2024SS,ya106020,NaN,NaN,78.5352,20,0.934197,Шляпин Алексей
702878,ya108000,"Ручка-толкатель для велосипеда 12""-16"" JKPB254",nan,2025SS,kari,лето,Летний спорт,Велосипеды,JKPB254,Принадлежности для спорта и активного отдыха,...,NaN,Бордачук Е. Спорт,2025SS,ya108000,NaN,NaN,155.2165,20,0.934197,Шляпин Алексей
702879,ya108010,Корзина для велосипеда голубая K13760,nan,2025SS,kari,лето,Летний спорт,Велосипеды,K13760,Принадлежности для спорта и активного отдыха,...,NaN,Бордачук Е. Спорт,2025SS,ya108010,NaN,NaN,95.6673,20,0.934197,Шляпин Алексей


In [17]:
df_union_reference = pd.merge(df_reference[['Артикул', 'Артикул OZ']], df_all_union, on="Артикул OZ", how='right')
# df_union_reference[df_union_reference['Артикул'].isna()]
df_union_reference

,Артикул,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт",Дата
0,y0806110,1607739332,823892917,301.0,1,2025-09-30
1,S7159817,2438516299,2310030768,2508.0,1,2025-09-30
2,W0257950,1627853566,1627855021,755.0,1,2025-09-30
3,W0257557,1627854626,1627852674,722.0,1,2025-09-30
4,W7587631,1627857305,1829020570,2084.0,1,2025-09-30
...,...,...,...,...,...,...
672296,M8205269,1267725462,351952384,5423.0,1,2025-10-30
672297,W8579567,2659157998,2659158129,3068.0,1,2025-10-30
672298,88109390,2810541646,2810540473,496.0,1,2025-10-30
672299,M5258910,1830569193,1830568337,1908.0,1,2025-10-30


In [18]:
df_union_reference['Продажи, ₽'] = pd.to_numeric(df_union_reference['Продажи, ₽'], errors='coerce')
df_union_reference['Заказы, шт'] = pd.to_numeric(df_union_reference['Заказы, шт'], errors='coerce')

df_union_agg = (
    df_union_reference
    .groupby(['Артикул', 'Дата'], as_index=False)
    .agg({
        'Артикул OZ': 'first',   # любой один из группы
        'SKU из объединенной карточки': 'first',   # любой один из группы
        'Продажи, ₽': 'sum',     # суммируем деньги
        'Заказы, шт': 'sum',     # суммируем заказы
    })
)
df_union_agg

,Артикул,Дата,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт"
0,00206110,2025-09-30,149484468,149484603,1706.0,2
1,00206110,2025-10-02,149484604,149484603,1706.0,2
2,00206110,2025-10-03,149484604,149484602,853.0,1
3,00206110,2025-10-04,149484603,149484602,853.0,1
4,00206110,2025-10-05,149484604,149484603,1706.0,2
...,...,...,...,...,...,...
293533,y9709050,2025-10-19,2804436405,2804437453,1269.0,1
293534,y9709050,2025-11-01,2804436405,2804437786,1379.0,1
293535,y9709050,2025-11-05,2804435772,2804436405,1379.0,1
293536,y9709050,2025-11-20,2804435772,2804436820,1091.0,1


In [19]:
df_union_reference.columns

Index(['Артикул', 'Артикул OZ', 'SKU из объединенной карточки', 'Продажи, ₽',
       'Заказы, шт', 'Дата'],
      dtype='object')

In [20]:
df_union_reference.drop_duplicates(subset=['Артикул','Дата'])

,Артикул,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт",Дата
0,y0806110,1607739332,823892917,301.0,1,2025-09-30
1,S7159817,2438516299,2310030768,2508.0,1,2025-09-30
2,W0257950,1627853566,1627855021,755.0,1,2025-09-30
3,W0257557,1627854626,1627852674,722.0,1,2025-09-30
4,W7587631,1627857305,1829020570,2084.0,1,2025-09-30
...,...,...,...,...,...,...
672280,W7519501,2438013668,1151436041,1992.0,1,2025-10-30
672286,310080J0,1775514447,1346903974,349.0,1,2025-10-30
672294,17007100,1649246458,1649246345,2135.0,1,2025-10-30
672296,M8205269,1267725462,351952384,5423.0,1,2025-10-30


In [21]:
df_all

,Артикул OZ,ID кампании,Инструмент,Место размещения,Дата
0,841850783,17708398,Оплата за клик,Поиск и рекомендации,2025-09-30
1,1628955757,17330786,Оплата за клик,Поиск и рекомендации,2025-09-30
2,2366656419,17576553,Оплата за клик,Поиск и рекомендации,2025-09-30
3,1151435994,17577580,Оплата за клик,Поиск и рекомендации,2025-09-30
4,1066494382,17558878,Оплата за клик,Поиск и рекомендации,2025-09-30
...,...,...,...,...,...
2308368,2310030361,18288265,Оплата за клик,NaN,2025-10-30
2308369,1649246458,18044269,Оплата за клик,NaN,2025-10-30
2308370,1267725462,17977104,Оплата за клик,NaN,2025-10-30
2308371,2659157998,18033552,Оплата за клик,NaN,2025-10-30


In [22]:
df_all=df_all.drop_duplicates(subset=['Дата','Артикул OZ'])

In [23]:
# df_funnel_copy = df_funnel.copy()
# df_funnel = df_result

In [24]:
# df_funnel_copy

In [25]:
# example = out[out['Артикул'] == 'u5501030']
# example=example[example['Дата'] == "2025-08-29"]
# example.to_excel('Example Ozon.xlsx')

In [26]:
# # На всякий случай выберем правильное имя артикула
# art_col = 'Артикул' if 'Артикул' in out.columns else 'Артикул WB'

# # Приведём дату к дню (если в данных есть время)
# out['Дата'] = pd.to_datetime(out['Дата'], errors='coerce').dt.normalize()

# # 1) Маска дубликатов по ключу (Дата, Артикул)
# mask = out.duplicated(subset=['Дата', art_col], keep=False)

# # 2) Сами строки-дубликаты (удобно смотреть)
# dups = out.loc[mask].sort_values(['Дата', art_col])

# # 3) Краткая сводка: какие пары и сколько раз встречаются (>1 — дубликаты)
# summary = (
#     out.groupby(['Дата', art_col])
#              .size().reset_index(name='Количество')
#              .query('Количество > 1')
# )

In [27]:
# dups#.dropna(subset=['Артикул'])

In [28]:
import numpy as np
# --- утилита: выбрать колонку расхода (на всякий случай) ---
SPEND_CANDIDATES = ['Расход, ₽', 'Расход, руб', 'Расход, Р', 'Расход']
def _pick_spend_col(df: pd.DataFrame) -> str:
    for c in SPEND_CANDIDATES:
        if c in df.columns:
            return c
    raise KeyError(f"Не найдена колонка расхода среди: {SPEND_CANDIDATES}")
# 10. Связать "Воронка" с "Справочник" БЕЗ дублирования
try:
    print("Начинаем создавать таблицу ВоронкаСправочник...")
    start_time = time.time()

    # --- 0) Валидация входа ---
    required_columns = ["Дата", "Артикул"]
    for col in required_columns:
        if col not in df_funnel.columns:
            raise ValueError(f"Отсутствует столбец '{col}' в df_funnel.")

    # --- 1) Нормализуем ключ (точно так же в обеих таблицах) ---
    df_funnel = df_funnel.copy()
    df_reference = df_reference.copy()

    df_funnel["Артикул"] = (df_funnel["Артикул"].fillna('')
                                              .astype(str).str.strip().str.upper()
                                              .str[:8])
    df_reference["Артикул"] = (df_reference["Артикул"].fillna('')
                                                    .astype(str).str.strip().str.upper()
                                                    .str[:8])

    # Приведём дату в единый формат (без времени)
    # df_funnel["Дата"] = pd.to_datetime(df_funnel["Дата"], errors="coerce").dt.normalize()

    # --- 2) Оставляем только нужные поля справочника ---
    reference_columns = [
        "Артикул", "Артикул OZ", "Наименование", "Коллекция", "Бренд", "Сезон", "Направление",
        "Розничный отдел", "Модель", "Группа", "Бизнес-группа", "Техсегмент",
        "Байер", "Две последние коллекции", "Основной артикул", "Себестоимость с НДС",
        "Процент выкупа", "НДС", "Ответственный за группу", "Группа для отчетов"
    ]
    # оставим только реально существующие колонки
    reference_columns = [c for c in reference_columns if c in df_reference.columns]
    df_reference_filtered = df_reference[["Артикул"] + [c for c in reference_columns if c != "Артикул"]].copy()

    # --- 3) Делаем справочник УНИКАЛЬНЫМ по Артикулу (one-row-per-Артикул) ---
    # Если есть дубликаты одного артикула — берём первую строку (можно заменить на приоритетное правило)
    dups = df_reference_filtered["Артикул"].duplicated(keep=False).sum()
    if dups:
        print(f"[INFO] В справочнике обнаружены дубликаты по 'Артикул' (после .str[:8]): {dups} строк.")
    ref_unique = (df_reference_filtered
                  .sort_values(["Артикул"])
                  .drop_duplicates(subset=["Артикул"], keep="first")
                  .reset_index(drop=True))

    # sanity: строго уникально
    assert not ref_unique["Артикул"].duplicated().any(), "ref_unique всё ещё содержит дубликаты Артикул"

    # --- 4) Контроль инвариантов расхода ДО merge ---
    try:
        spend_col = _pick_spend_col(df_funnel)
    except KeyError:
        spend_col = None

    if spend_col:
        before_total = pd.to_numeric(df_funnel[spend_col], errors='coerce').sum()
        before_by_date = (df_funnel.groupby("Дата", as_index=False)[spend_col]
                                   .sum(min_count=1)
                                   .rename(columns={spend_col: "Расход_до"}))
    else:
        print("[INFO] В df_funnel нет колонки расхода — инварианты по расходу не проверяем.")

    # --- 5) LEFT-merge строго m:1 (исключает размножение строк) ---
    df_funnel_reference = pd.merge(
        df_funnel,
        ref_unique,
        on="Артикул",
        how="left",
        validate="m:1",     # если справа снова появятся дубликаты — упадёт сразу
        indicator=False
    )

    # Ничего НЕ дропаем из df_funnel_reference! (drop_duplicates ломает суммы)

    # --- 6) Контроль ПОСЛЕ merge (сумма не должна измениться ни по датам, ни итого) ---
    if spend_col:
        after_total = pd.to_numeric(df_funnel_reference[spend_col], errors='coerce').sum()
        after_by_date = (df_funnel_reference.groupby("Дата", as_index=False)[spend_col]
                                          .sum(min_count=1)
                                          .rename(columns={spend_col: "Расход_после"}))
        check = before_by_date.merge(after_by_date, on="Дата", how="outer").fillna(0)
        drift = check.loc[~np.isclose(check["Расход_до"], check["Расход_после"], rtol=1e-9, atol=1e-6)]

        print(f"[CHECK] Общая сумма расхода: до={before_total:,.2f} | после={after_total:,.2f}")
        if len(drift):
            print("[WARN] Обнаружены расхождения по датам (первые 10):")
            print(drift.head(10))
            # Если тут расхождения — значит есть иная проблема (например, обрезка .str[:8] склеила разные артикула)
    else:
        after_total = None

    # --- 7) Формат вывода даты (если нужен именно date без времени) ---
    # df_funnel_reference["Дата"] = pd.to_datetime(df_funnel_reference["Дата"]).dt.date

    # Итог
    print("Первые 5 строк таблицы ВоронкаСправочник:")
    print(df_funnel_reference.head())

    elapsed_time = time.time() - start_time
    print(f"Таблица ВоронкаСправочник успешно создана. Время выполнения: {elapsed_time:.2f} c.")

except Exception as e:
    print(f"Ошибка при создании таблицы ВоронкаСправочник: {e}")

Начинаем создавать таблицу ВоронкаСправочник...
[INFO] В справочнике обнаружены дубликаты по 'Артикул' (после .str[:8]): 607112 строк.
[CHECK] Общая сумма расхода: до=502,644,526.83 | после=502,644,526.83
Первые 5 строк таблицы ВоронкаСправочник:
         Дата   Артикул ТипАктивности  Показы, всего  \
0  2025-10-01  31306230      Органика             34   
1  2025-10-01  17206150      Органика             64   
2  2025-10-01  17206200      Органика             14   
3  2025-10-01  17106050      Органика             40   
4  2025-10-01  17206190      Органика             80   

   Показы на карточке товара  Показы в поиске и каталоге  \
0                          8                          12   
1                          4                          48   
2                          0                           0   
3                          3                          18   
4                          7                          46   

   Позиция в поиске и каталоге  В корзину, всего  Заказ

In [29]:
df_funnel_reference

,Дата,Артикул,ТипАктивности,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,...,Бизнес-группа,Техсегмент,Байер,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов
0,2025-10-01,31306230,Органика,34,8,12,192.83,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,31306230,395.5555,0.964286,10.0,Никонорова Алина,Пляжная одежда
1,2025-10-01,17206150,Органика,64,4,48,234.82,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,17206150,250.2334,1.000000,10.0,Зиннуров Ильнур,Детская одежда
2,2025-10-01,17206200,Органика,14,0,0,NaN,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,17206200,250.2334,0.894737,10.0,Зиннуров Ильнур,Детская одежда
3,2025-10-01,17106050,Органика,40,3,18,142.89,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,17106050,220.2081,0.800000,10.0,Зиннуров Ильнур,Детская одежда
4,2025-10-01,17206190,Органика,80,7,46,240.15,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,17206190,208.4333,1.000000,10.0,Зиннуров Ильнур,Детская одежда
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3080762,2025-10-31,C0101130,Органика,1,0,0,NaN,0,0,0,...,Аксессуары,NaN,Коновалова А.,2021AW,c0101130,NaN,0.930547,20.0,Никонорова Алина,NaN
3080763,2025-10-31,C1101020,Органика,3,1,1,NaN,0,0,0,...,Аксессуары,NaN,Коновалова А.,2021AW,c1101020,NaN,0.930547,20.0,Никонорова Алина,NaN
3080764,2025-10-31,U4906230,Органика,1,0,0,NaN,0,0,0,...,Игрушки,NaN,Евтикова Н.А.,"2024AW,2024SS",u4906230,303.7363,0.931701,10.0,Валиков Никита,Игрушки
3080765,2025-10-31,,Органика,1,0,0,NaN,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-11-17']['Расход, ₽'].sum()

np.float64(12742959.61)

In [31]:
df_funnel_reference

,Дата,Артикул,ТипАктивности,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,...,Бизнес-группа,Техсегмент,Байер,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов
0,2025-10-01,31306230,Органика,34,8,12,192.83,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,31306230,395.5555,0.964286,10.0,Никонорова Алина,Пляжная одежда
1,2025-10-01,17206150,Органика,64,4,48,234.82,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,17206150,250.2334,1.000000,10.0,Зиннуров Ильнур,Детская одежда
2,2025-10-01,17206200,Органика,14,0,0,NaN,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,17206200,250.2334,0.894737,10.0,Зиннуров Ильнур,Детская одежда
3,2025-10-01,17106050,Органика,40,3,18,142.89,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,17106050,220.2081,0.800000,10.0,Зиннуров Ильнур,Детская одежда
4,2025-10-01,17206190,Органика,80,7,46,240.15,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,17206190,208.4333,1.000000,10.0,Зиннуров Ильнур,Детская одежда
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3080762,2025-10-31,C0101130,Органика,1,0,0,NaN,0,0,0,...,Аксессуары,NaN,Коновалова А.,2021AW,c0101130,NaN,0.930547,20.0,Никонорова Алина,NaN
3080763,2025-10-31,C1101020,Органика,3,1,1,NaN,0,0,0,...,Аксессуары,NaN,Коновалова А.,2021AW,c1101020,NaN,0.930547,20.0,Никонорова Алина,NaN
3080764,2025-10-31,U4906230,Органика,1,0,0,NaN,0,0,0,...,Игрушки,NaN,Евтикова Н.А.,"2024AW,2024SS",u4906230,303.7363,0.931701,10.0,Валиков Никита,Игрушки
3080765,2025-10-31,,Органика,1,0,0,NaN,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
df_union_agg#.drop_duplicates(subset=['Артикул OZ', 'SKU из объединенной карточки'])

,Артикул,Дата,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт"
0,00206110,2025-09-30,149484468,149484603,1706.0,2
1,00206110,2025-10-02,149484604,149484603,1706.0,2
2,00206110,2025-10-03,149484604,149484602,853.0,1
3,00206110,2025-10-04,149484603,149484602,853.0,1
4,00206110,2025-10-05,149484604,149484603,1706.0,2
...,...,...,...,...,...,...
293533,y9709050,2025-10-19,2804436405,2804437453,1269.0,1
293534,y9709050,2025-11-01,2804436405,2804437786,1379.0,1
293535,y9709050,2025-11-05,2804435772,2804436405,1379.0,1
293536,y9709050,2025-11-20,2804435772,2804436820,1091.0,1


In [33]:
# объединяем с df_funnel
df_result = df_funnel_reference.merge(
    df_all,
    on=["Дата", "Артикул OZ"],
    how="left"
)

df_result = df_result.merge(
    df_union_agg,
    on=["Дата", "Артикул OZ"],
    how="left"
)

# Трафарет
mask = (
    ((df_result['Инструмент'] == 'Оплата за клик') &
    (df_result['Место размещения'] == 'Поиск и рекомендации')) |
    (df_result['ТипАктивности'] == 'Оплата за клик')
)
df_result.loc[mask, 'ТипАктивности'] = 'Трафарет'

# Вывод в топ
mask = (
    ((df_result['Инструмент'] == 'Оплата за клик') & 
     (df_result['Место размещения'] == 'Поиск')) |
    (df_result['ТипАктивности'] == 'ТОП')
)
df_result.loc[mask, 'ТипАктивности'] = 'Вывод в топ'

# Органика
mask = (
    ((df_result['Расход, ₽'] == 0) | 
     (df_result['Расход, ₽'] == 0.0))
)
df_result.loc[mask, 'ТипАктивности'] = 'Органика'

In [34]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-11-17']['Расход, ₽'].sum()

np.float64(12742959.61)

In [35]:
df_result[df_result['Дата'] == '2025-11-17']['Расход, ₽'].sum()

np.float64(12742959.61)

In [36]:
df_funnel_reference

,Дата,Артикул,ТипАктивности,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,...,Бизнес-группа,Техсегмент,Байер,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов
0,2025-10-01,31306230,Органика,34,8,12,192.83,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,31306230,395.5555,0.964286,10.0,Никонорова Алина,Пляжная одежда
1,2025-10-01,17206150,Органика,64,4,48,234.82,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,17206150,250.2334,1.000000,10.0,Зиннуров Ильнур,Детская одежда
2,2025-10-01,17206200,Органика,14,0,0,NaN,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,17206200,250.2334,0.894737,10.0,Зиннуров Ильнур,Детская одежда
3,2025-10-01,17106050,Органика,40,3,18,142.89,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,17106050,220.2081,0.800000,10.0,Зиннуров Ильнур,Детская одежда
4,2025-10-01,17206190,Органика,80,7,46,240.15,0,0,0,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,17206190,208.4333,1.000000,10.0,Зиннуров Ильнур,Детская одежда
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3080762,2025-10-31,C0101130,Органика,1,0,0,NaN,0,0,0,...,Аксессуары,NaN,Коновалова А.,2021AW,c0101130,NaN,0.930547,20.0,Никонорова Алина,NaN
3080763,2025-10-31,C1101020,Органика,3,1,1,NaN,0,0,0,...,Аксессуары,NaN,Коновалова А.,2021AW,c1101020,NaN,0.930547,20.0,Никонорова Алина,NaN
3080764,2025-10-31,U4906230,Органика,1,0,0,NaN,0,0,0,...,Игрушки,NaN,Евтикова Н.А.,"2024AW,2024SS",u4906230,303.7363,0.931701,10.0,Валиков Никита,Игрушки
3080765,2025-10-31,,Органика,1,0,0,NaN,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
df_result.drop(columns=['Артикул_y'], inplace=True)
df_result.rename(columns={'Артикул_x':'Артикул'}, inplace=True)

In [38]:
df_result['ТипАктивности'].unique()

array(['Органика', 'Трафарет', 'Оплата за заказ', 'Вывод в топ'],
      dtype=object)

In [39]:
df_result.columns

Index(['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего',
       'Показы на карточке товара', 'Показы в поиске и каталоге',
       'Позиция в поиске и каталоге', 'В корзину, всего', 'Заказано товаров',
       'Отменено товаров', 'Доставлено товаров', 'Возвращено товаров',
       'Заказано на сумму', 'В корзину из карточки товара', 'Выкупили ШТ',
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена', 'Артикул OZ',
       'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление',
       'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент',
       'Байер', 'Две последние коллекции', 'Основной артикул',
       'Себестоимость с НДС', 'Процент выкупа', 'НДС',
       'Ответственный за группу', 'Группа для отчетов', 'ID кампании',
       'Инструмент', 'Место размещения', 'SKU из объединенной карточки',
       'Продажи, ₽', 'Заказы, шт'],
      dtype='object')

In [40]:
df_result.rename(columns={'Продажи, ₽':'Ассоциированные заказы, руб', 'Заказы, шт':'Ассоциированные заказы, шт'}, inplace=True)

In [41]:
# df_funnel_reference_copy = df_funnel_reference.copy()
df_funnel_reference = df_result

In [42]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-11-17']['Расход, ₽'].sum()

np.float64(12742959.61)

In [43]:
df_funnel_reference.columns

Index(['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего',
       'Показы на карточке товара', 'Показы в поиске и каталоге',
       'Позиция в поиске и каталоге', 'В корзину, всего', 'Заказано товаров',
       'Отменено товаров', 'Доставлено товаров', 'Возвращено товаров',
       'Заказано на сумму', 'В корзину из карточки товара', 'Выкупили ШТ',
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена', 'Артикул OZ',
       'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление',
       'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент',
       'Байер', 'Две последние коллекции', 'Основной артикул',
       'Себестоимость с НДС', 'Процент выкупа', 'НДС',
       'Ответственный за группу', 'Группа для отчетов', 'ID кампании',
       'Инструмент', 'Место размещения', 'SKU из объединенной карточки',
       'Ассоциированные заказы, руб', 'Ассоциированные заказы, шт'

In [44]:
funnel_columns = ['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена'
       ]
funnel_columns_all = [
    'Дата', 'Артикул', 'ТипАктивности', 'Показы, всего',
    'Показы на карточке товара', 'Показы в поиске и каталоге',
    'Позиция в поиске и каталоге', 'В корзину, всего',
    'Заказано товаров', 'Отменено товаров', 'Доставлено товаров',
    'Возвращено товаров', 'Заказано на сумму',
    'В корзину из карточки товара', 'Выкупили ШТ',
    'Расход, ₽', 'Рекламные заказано на сумму',
    'Рекламные заказано товаров', 'Рекламные показы',
    'Рекламные показы на карточке товара', 'Цена',
    'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон',
    'Направление', 'Розничный отдел', 'Модель', 'Группа',
    'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции',
    'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС',
    'Ответственный за группу', 'Группа для отчетов',
    'ID кампании', 'Инструмент', 'Место размещения'
]
funnel_columns_widing = ['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена'
       ]

In [45]:
# import pandas as pd
# import numpy as np

# # --- 0) подготовка
# df = df_funnel_reference.copy()

# # переименовать тип активности "ТОП" -> "Вывод в топ"
# df.loc[df['ТипАктивности'].eq('ТОП'), 'ТипАктивности'] = 'Вывод в топ'

# # полный список типов (жёстко фиксируем порядок и наличие)
# ALL_TYPES = ['Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Спецразмещение', 'Органика']

# # ключи и метрики
# key_cols = ['Дата', 'Артикул']
# # всё, что стоит в таблице ПОСЛЕ "ТипАктивности", считаем метриками
# cols = funnel_columns
# metric_cols = cols[cols.index('ТипАктивности') + 1 :]

# print(metric_cols)

# # привести метрики к числам (на случай строк/пробелов)
# for c in metric_cols:
#     df[c] = pd.to_numeric(df[c], errors='coerce')

# # --- 1) агрегация по (Дата, Артикул, ТипАктивности)
# g = (df
#      .groupby(key_cols + ['ТипАктивности'], as_index=False)[metric_cols]
#      .sum(min_count=1)
# )

# # --- 2) «широкая» таблица метрик с префиксами <Тип>_<Метрика>
# wide_metrics = g.pivot_table(
#     index=key_cols,
#     columns='ТипАктивности',
#     values=metric_cols,
#     aggfunc='sum',
#     fill_value=0
# )

# # гарантируем наличие ВСЕХ типов и ВСЕХ метрик (даже если их не было в данных)
# full_cols = pd.MultiIndex.from_product([metric_cols, ALL_TYPES])
# wide_metrics = wide_metrics.reindex(columns=full_cols, fill_value=0)

# # имена колонок: "Тип_Метрика"
# wide_metrics.columns = [f'{act}_{met}' for met, act in wide_metrics.columns.to_flat_index()]
# wide_metrics = wide_metrics.reset_index()

# # --- 3) бинарные признаки наличия типа активности (1/0) по каждой паре (Дата, Артикул)
# presence = (
#     df.groupby(key_cols + ['ТипАктивности']).size()
#       .reset_index(name='n')
#       .pivot(index=key_cols, columns='ТипАктивности', values='n')
#       .reindex(columns=ALL_TYPES, fill_value=0)
#       .gt(0).astype(int)  # 1 если был хотя бы один ряд данного типа
#       .reset_index()
# )

# # --- 4) объединяем метрики и бинарные признаки
# out = (wide_metrics
#        .merge(presence, on=key_cols, how='left')
#        .fillna(0)
# )

# # --- 5) итоговые столбцы БЕЗ префиксов = сумма по всем типам
# for met in metric_cols:
#     to_sum = [f'{t}_{met}' for t in ALL_TYPES if f'{t}_{met}' in out.columns]
#     if to_sum:
#         out[met] = out[to_sum].sum(axis=1)

# # --- 6) порядок колонок: ключи → бинарные типы → для каждой метрики столбцы по типам → итог по метрике
# ordered = key_cols + ALL_TYPES[:]  # бинарные столбцы имеют те же имена, что и типы
# for met in metric_cols:
#     ordered += [f'{t}_{met}' for t in ALL_TYPES]
#     ordered += [met]
# # оставим только реально существующие (вдруг каких-то метрик не было)
# ordered = [c for c in ordered if c in out.columns]

# out = out[ordered]

# # результат в переменной `out`
# # одна строка на (Дата, Артикул), колоноки вида:
# # Дата | Артикул | Вывод в топ | Трафарет | ... | Вывод в топ_Показы, всего | Трафарет_Показы, всего | ... | Показы, всего | ...


In [46]:
import numpy as np
import pandas as pd

def build_funnel_wide(
    df_raw: pd.DataFrame,
    funnel_columns: list,
    all_types=('Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика'),
    infer_organic_by_zero_spend=False,
    spend_col='Расход, ₽',
    extra_agg='first',          # 'first' | 'join'
    extra_join_sep=' | ',
    check_spend_invariance=True,
    atol=1e-6, rtol=1e-9
):
    """
    Склеивает строки по (Дата, Артикул), раскладывает метрики по типам активности,
    добавляет ИТОГИ, которые считаются напрямую из исходника по ключу (Дата, Артикул).
    Благодаря этому 'Расход, ₽' (и прочие итоги) сохраняют исходные значения.
    """

    # ---- 0) Исходная выборка для расчётов (только нужные колонки) ----
    cols_present = [c for c in funnel_columns if c in df_raw.columns]
    df = df_raw.loc[:, cols_present].copy()

    # ---- 1) Нормализуем тип активности ----
    type_col = 'ТипАктивности'
    df[type_col] = df[type_col].replace({'ТОП': 'Вывод в топ'})
    if infer_organic_by_zero_spend and spend_col in df.columns:
        m0 = pd.to_numeric(df[spend_col], errors='coerce').fillna(0).eq(0)
        df.loc[m0, type_col] = 'Органика'

    # ---- 2) Ключи/метрики и приведение типов ----
    key_cols = ['Дата', 'Артикул']
    met_start = funnel_columns.index(type_col) + 1
    metric_cols = [c for c in funnel_columns[met_start:] if c in df.columns]

    # аккуратно приводим метрики
    for c in metric_cols:
        if c == spend_col:
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('float64')   # спенд в float64
        else:
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')

    # ---- 3) ИТОГИ БЕЗ ПРЕФИКСОВ (ИСТИНА) по (Дата, Артикул) ----
    totals_df = (df.groupby(key_cols, as_index=False)[metric_cols]
                   .sum(min_count=1))   # если где-то все NaN, останется NaN; это корректно

    # ---- 4) Префиксные метрики по типам ----
    g = (df.groupby(key_cols + [type_col], as_index=False)[metric_cols]
           .sum(min_count=1))

    # Дополним all_types тем, что реально встретилось
    types_present = g[type_col].dropna().unique().tolist()
    all_types = list(dict.fromkeys(list(all_types) + [t for t in types_present if t not in all_types]))

    # Полная база ключей = все пары (Дата, Артикул), которые встречаются в исходнике
    base = (df[key_cols].drop_duplicates()
                    .set_index(key_cols)
                    .sort_index())

    # Сформируем блоки префиксных метрик и флаги наличия типов
    metric_blocks, flag_blocks = [], []
    for t in all_types:
        sub = g[g[type_col] == t].set_index(key_cols)

        if sub.empty:
            # пустой тип → нули на всю базу
            sub_metrics = pd.DataFrame(
                0.0, index=base.index,
                columns=[f'{t}_{m}' for m in metric_cols],
                dtype='float32'
            )
        else:
            sub_metrics = (sub[metric_cols]
                           .rename(columns={m: f'{t}_{m}' for m in metric_cols})
                           .reindex(base.index, fill_value=0.0))

            # типы данных: спенд оставляем float64
            for col in sub_metrics.columns:
                if col.endswith(spend_col):
                    sub_metrics[col] = sub_metrics[col].astype('float64')
                else:
                    sub_metrics[col] = sub_metrics[col].astype('float32')

        metric_blocks.append(sub_metrics)

        # бинарный флаг присутствия типа (на уровне ключа)
        flag = pd.Series(1, index=sub.index, name=t) if not sub.empty else pd.Series(0, index=base.index, name=t)
        flag_blocks.append(flag.reindex(base.index, fill_value=0).astype('int8'))

    metrics_block = pd.concat(metric_blocks, axis=1)
    flags_block   = pd.concat(flag_blocks, axis=1)

    # ---- 5) СБОРКА CORE: ключи + флаги + префиксные метрики + ИТОГИ ИЗ totals_df ----
    core = pd.concat(
        [
            base.reset_index(),
            flags_block.reset_index(drop=True),
            metrics_block.reset_index(drop=True)
        ],
        axis=1
    )

    # присоединяем ИТОГИ (истина) строго m:1
    core = core.merge(totals_df, on=key_cols, how='left', validate='m:1')

    # ---- 6) ДОП. колонки из df_raw (НЕ участвуют в расчётах) ----
    exclude = set(key_cols + [type_col] + metric_cols)
    extra_cols = [c for c in df_raw.columns if c not in exclude]
    if extra_cols:
        if extra_agg == 'first':
            dims_block = (df_raw[key_cols + extra_cols]
                            .sort_values(key_cols)
                            .groupby(key_cols, as_index=False)
                            .first())
        elif extra_agg == 'join':
            def _join_unique(s):
                v = pd.unique(s.dropna().astype(str))
                return extra_join_sep.join(v) if len(v) else np.nan
            dims_block = (df_raw[key_cols + extra_cols]
                            .groupby(key_cols, as_index=False)
                            .agg({c: _join_unique for c in extra_cols}))
        else:
            raise ValueError("extra_agg должен быть 'first' или 'join'")

        out = core.merge(dims_block, on=key_cols, how='left', validate='m:1')
    else:
        out = core

    # ---- 7) Проверка инварианта для 'Расход, ₽' (опционально) ----
    if check_spend_invariance and (spend_col in totals_df.columns):
        base_sp = (totals_df.groupby(key_cols, as_index=False)[spend_col].sum(min_count=1)
                             .rename(columns={spend_col: '__base__'}))
        after_sp = (out.groupby(key_cols, as_index=False)[spend_col].sum(min_count=1)
                          .rename(columns={spend_col: '__after__'}))
        chk = base_sp.merge(after_sp, on=key_cols, how='outer').fillna(0)
        bad = chk.loc[~np.isclose(chk['__base__'], chk['__after__'], rtol=rtol, atol=atol)]
        if not bad.empty:
            print("[WARN] Инвариант по 'Расход, ₽' нарушен для некоторых ключей (первые 10):")
            print(bad.head(10))

    # ---- 8) Порядок колонок: ключи → доп.колонки → флаги → префиксные метрики → ИТОГИ ----
    ordered = []
    ordered += key_cols
    ordered += [c for c in df_raw.columns if (c in out.columns and c not in key_cols and c not in ([type_col] + metric_cols))]
    ordered += [t for t in all_types if t in out.columns]

    for m in metric_cols:
        # префиксные
        ordered += [f'{t}_{m}' for t in all_types if f'{t}_{m}' in out.columns]
    # ИТОГИ (без префикса) — в самом конце блоком в исходном порядке
    ordered += [m for m in metric_cols if m in out.columns]

    out = out[[c for c in ordered if c in out.columns]].copy()
    return out


In [47]:
out = build_funnel_wide(df_raw=df_funnel_reference, funnel_columns=funnel_columns_widing)
out

,Дата,Артикул,Артикул OZ,Наименование,Коллекция,Бренд,Сезон,Направление,Розничный отдел,Модель,...,Возвращено товаров,Заказано на сумму,В корзину из карточки товара,Выкупили ШТ,"Расход, ₽",Рекламные заказано на сумму,Рекламные заказано товаров,Рекламные показы,Рекламные показы на карточке товара,Цена
0,2025-10-01,00006020,149393764,Балетки женские 298-9CK,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,298-9CK,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,569.0
1,2025-10-01,00006070,149393762,Балетки женские ZS17S-10AKK,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,ZS17S-10AKK,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,609.0
2,2025-10-01,00006080,149390933,Балетки женские ZS189-7K,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,ZS189-7K,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,690.0
3,2025-10-01,00006090,149390951,Балетки женские ZS71-1BK,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,ZS71-1BK,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,550.0
4,2025-10-01,00006110,149393771,Балетки женские 299-8K,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,299-8K,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,495.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3079403,2025-12-01,Y9808110,1997933020,Шорты мужские A85512-2,2025SS,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,A85512-2,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1810.0
3079404,2025-12-01,Y9808120,1997932758,Шорты мужские A85512-3,2025SS,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,A85512-3,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1910.0
3079405,2025-12-01,Y9808130,1934644709,Шорты мужские SS25C2020,2025SS,kari,"весна, лето",Одежда для мужчин,"Брюки, шорты мужские",SS25C2020,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1913.0
3079406,2025-12-01,Y9808140,1950208470,Шорты мужские SS25C2021,2025SS,kari,"весна, лето",Одежда для мужчин,"Брюки, шорты мужские",SS25C2021,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1768.0


In [48]:
out.columns

Index(['Дата', 'Артикул', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд',
       'Сезон', 'Направление', 'Розничный отдел', 'Модель',
       ...
       'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 'Расход, ₽',
       'Рекламные заказано на сумму', 'Рекламные заказано товаров',
       'Рекламные показы', 'Рекламные показы на карточке товара', 'Цена'],
      dtype='object', length=121)

In [49]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-11-17']['Расход, ₽'].sum()

np.float64(12742959.61)

In [50]:
out[out['Дата'] == '2025-11-17']['Расход, ₽'].sum()

np.float64(12742959.609999998)

In [51]:
df_funnel_reference = out

In [52]:
df_funnel_reference.columns

Index(['Дата', 'Артикул', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд',
       'Сезон', 'Направление', 'Розничный отдел', 'Модель',
       ...
       'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 'Расход, ₽',
       'Рекламные заказано на сумму', 'Рекламные заказано товаров',
       'Рекламные показы', 'Рекламные показы на карточке товара', 'Цена'],
      dtype='object', length=121)

In [53]:
df_funnel_reference

,Дата,Артикул,Артикул OZ,Наименование,Коллекция,Бренд,Сезон,Направление,Розничный отдел,Модель,...,Возвращено товаров,Заказано на сумму,В корзину из карточки товара,Выкупили ШТ,"Расход, ₽",Рекламные заказано на сумму,Рекламные заказано товаров,Рекламные показы,Рекламные показы на карточке товара,Цена
0,2025-10-01,00006020,149393764,Балетки женские 298-9CK,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,298-9CK,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,569.0
1,2025-10-01,00006070,149393762,Балетки женские ZS17S-10AKK,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,ZS17S-10AKK,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,609.0
2,2025-10-01,00006080,149390933,Балетки женские ZS189-7K,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,ZS189-7K,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,690.0
3,2025-10-01,00006090,149390951,Балетки женские ZS71-1BK,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,ZS71-1BK,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,550.0
4,2025-10-01,00006110,149393771,Балетки женские 299-8K,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,299-8K,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,495.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3079403,2025-12-01,Y9808110,1997933020,Шорты мужские A85512-2,2025SS,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,A85512-2,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1810.0
3079404,2025-12-01,Y9808120,1997932758,Шорты мужские A85512-3,2025SS,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,A85512-3,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1910.0
3079405,2025-12-01,Y9808130,1934644709,Шорты мужские SS25C2020,2025SS,kari,"весна, лето",Одежда для мужчин,"Брюки, шорты мужские",SS25C2020,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1913.0
3079406,2025-12-01,Y9808140,1950208470,Шорты мужские SS25C2021,2025SS,kari,"весна, лето",Одежда для мужчин,"Брюки, шорты мужские",SS25C2021,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1768.0


In [54]:
# 11. Связать "ВоронкаСправочник" с "Остатки с дистрибуцией"
try:
    print("Начинаем создавать таблицу ДБбезПризнаков...")
    start_time = time.time()  # Запускаем таймер
    df_final_db = pd.merge(df_funnel_reference, df_stock_with_distribution, left_on=["Дата", "Артикул"], right_on=["Дата", "Артикул"], how="left")
    df_final_db = format_date_column(df_final_db, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБбезПризнаков:")
    print(df_final_db.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБбезПризнаков успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБбезПризнаков: {e}")

Начинаем создавать таблицу ДБбезПризнаков...
Первые 5 строк таблицы ДБбезПризнаков:
         Дата   Артикул Артикул OZ                 Наименование Коллекция  \
0  2025-10-01  00006020  149393764      Балетки женские 298-9CK    2019SS   
1  2025-10-01  00006070  149393762  Балетки женские ZS17S-10AKK    2019SS   
2  2025-10-01  00006080  149390933     Балетки женские ZS189-7K    2019SS   
3  2025-10-01  00006090  149390951     Балетки женские ZS71-1BK    2019SS   
4  2025-10-01  00006110  149393771       Балетки женские 299-8K    2019SS   

        Бренд            Сезон    Направление Розничный отдел       Модель  \
0  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь      298-9CK   
1  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь  ZS17S-10AKK   
2  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь     ZS189-7K   
3  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь     ZS71-1BK   
4  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь  

In [55]:
df_final_db.columns

Index(['Дата', 'Артикул', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд',
       'Сезон', 'Направление', 'Розничный отдел', 'Модель',
       ...
       'В корзину из карточки товара', 'Выкупили ШТ', 'Расход, ₽',
       'Рекламные заказано на сумму', 'Рекламные заказано товаров',
       'Рекламные показы', 'Рекламные показы на карточке товара', 'Цена',
       'Остаток Агрегатора', 'Дистрибуция'],
      dtype='object', length=123)

In [56]:
# 5. Получить данные из файла !!!_Признаки для артикула и даты для Озон
try:
    print("Начинаем получать данные для Признаков...")
    start_time = time.time()  # Запускаем таймер
    file_path_features = os.path.join(FOLDER_PATH_FEATURES, "!!!_Признаки для артикула и даты для Озон.xlsx")
    if os.path.exists(file_path_features):
        df_item_features = pd.read_excel(file_path_features, sheet_name="Признаки для артикула", dtype=str, engine="calamine")
        df_date_features = pd.read_excel(file_path_features, sheet_name="Признаки для дат", dtype={0: "datetime64[ns]", **{i: str for i in range(1, 6)}}, engine="calamine")

        # Обработка ошибок
        df_item_features = handle_errors(df_item_features)
        df_date_features = handle_errors(df_date_features)

        # Форматирование даты
        df_date_features = format_date_column(df_date_features, 'Дата')

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки артикула:")
        print(df_item_features.head())

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки дат:")
        print(df_date_features.head())

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Признаков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл '!!!_Признаки для артикула и даты для Озон.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Признаков: {e}")

Начинаем получать данные для Признаков...
Первые 5 строк таблицы Признаки артикула:
    Артикул Признак Артикула 1 Признак Артикула 2 Признак Артикула 3  \
0  00001851                NaN                NaN                NaN   
1  00001852                NaN                NaN                NaN   
2  00001855                NaN                NaN                NaN   
3  00001856                NaN                NaN                NaN   
4  00001931                NaN                NaN                NaN   

  Признак Артикула 4 Признак Артикула 5  
0                NaN                NaN  
1                NaN                NaN  
2                NaN                NaN  
3                NaN                NaN  
4                NaN                NaN  
Первые 5 строк таблицы Признаки дат:
Empty DataFrame
Columns: [Дата, Признак Даты 1, Признак Даты 2, Признак Даты 3, Признак Даты 4, Признак Даты 5]
Index: []
Данные для Признаков успешно сохранены. Время выполнения: 0 часа(ов) 0 м

In [57]:
# 12. Связать "ДБбезПризнаков" с "Признаки для артикула"
try:
    print("Начинаем создавать таблицу ДБсПризнакамиАртикула...")
    start_time = time.time()  # Запускаем таймер
    df_final_db_item_features = pd.merge(df_final_db, df_item_features, left_on=["Артикул"], right_on=["Артикул"], how="left")
    df_final_db_item_features = format_date_column(df_final_db_item_features, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБсПризнакамиАртикула:")
    print(df_final_db_item_features.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБсПризнакамиАртикула успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнакамиАртикула: {e}")

Начинаем создавать таблицу ДБсПризнакамиАртикула...
Первые 5 строк таблицы ДБсПризнакамиАртикула:
         Дата   Артикул Артикул OZ                 Наименование Коллекция  \
0  2025-10-01  00006020  149393764      Балетки женские 298-9CK    2019SS   
1  2025-10-01  00006070  149393762  Балетки женские ZS17S-10AKK    2019SS   
2  2025-10-01  00006080  149390933     Балетки женские ZS189-7K    2019SS   
3  2025-10-01  00006090  149390951     Балетки женские ZS71-1BK    2019SS   
4  2025-10-01  00006110  149393771       Балетки женские 299-8K    2019SS   

        Бренд            Сезон    Направление Розничный отдел       Модель  \
0  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь      298-9CK   
1  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь  ZS17S-10AKK   
2  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь     ZS189-7K   
3  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь     ZS71-1BK   
4  T.TACCARDI  лето (закрытое)  Женская обувь   Ж

In [58]:
# 13. Связать "ДБсПризнакамиАртикула" с "Признаки для дат"
try:
    print("Начинаем создавать таблицу ДБсПризнаками...")
    start_time = time.time()
    df_final_db_all_features = pd.merge(df_final_db_item_features, df_date_features, on="Дата", how="left")
    df_final_db_all_features = format_date_column(df_final_db_all_features, 'Дата')

    print("Первые 5 строк таблицы ДБсПризнаками:")
    print(df_final_db_all_features.head())

    # Сохранение финальной таблицы
    # df_final_db_all_features.to_csv(os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon_New.csv"), index=False)

    elapsed_time = time.time() - start_time
    print(f"Таблица ДБсПризнаками успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнаками: {e}")

Начинаем создавать таблицу ДБсПризнаками...
Первые 5 строк таблицы ДБсПризнаками:
         Дата   Артикул Артикул OZ                 Наименование Коллекция  \
0  2025-10-01  00006020  149393764      Балетки женские 298-9CK    2019SS   
1  2025-10-01  00006070  149393762  Балетки женские ZS17S-10AKK    2019SS   
2  2025-10-01  00006080  149390933     Балетки женские ZS189-7K    2019SS   
3  2025-10-01  00006090  149390951     Балетки женские ZS71-1BK    2019SS   
4  2025-10-01  00006110  149393771       Балетки женские 299-8K    2019SS   

        Бренд            Сезон    Направление Розничный отдел       Модель  \
0  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь      298-9CK   
1  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь  ZS17S-10AKK   
2  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь     ZS189-7K   
3  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь     ZS71-1BK   
4  T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь    

In [59]:
df_final_db_all_features[df_final_db_all_features['Дата'] == '2025-11-17']['Расход, ₽'].sum()

np.float64(12742959.609999998)

In [60]:
import numpy as np
# df — ваша широкая таблица с флагами типов (1/0)
type_order_paid = ['Вывод в топ', 'Трафарет', 'Оплата за заказ']
paid_cols = [c for c in type_order_paid if c in df_final_db_all_features.columns]  # на случай отсутствующих

# Матрица флагов платных типов
flags = df_final_db_all_features[paid_cols].fillna(0).astype('uint8').to_numpy()
labels = np.array(paid_cols, dtype=object)

# Собираем подписи для платных комбинаций
combo = ['/'.join(labels[row.astype(bool)]) if row.any() else '' for row in flags]
df_final_db_all_features['ТипАктивности'] = combo

# Если есть только органика — подставим "Органика"
if 'Органика' in df_final_db_all_features.columns:
    only_org = df_final_db_all_features['Органика'].fillna(0).astype('uint8').eq(1) & (flags.sum(axis=1) == 0)
    df_final_db_all_features.loc[only_org, 'ТипАктивности'] = 'Органика'

# Пустые — на "—"
df_final_db_all_features['ТипАктивности'] = df_final_db_all_features['ТипАктивности'].replace('', '—')

In [61]:
df_final_db_all_features[df_final_db_all_features['Дата'] == '2025-11-17']['Расход, ₽'].sum()

np.float64(12742959.609999998)

In [62]:
print(list(df_final_db_all_features.columns))

['Дата', 'Артикул', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС', 'Ответственный за группу', 'Группа для отчетов', 'ID кампании', 'Инструмент', 'Место размещения', 'SKU из объединенной карточки', 'Ассоциированные заказы, руб', 'Ассоциированные заказы, шт', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика', 'Вывод в топ_Показы, всего', 'Трафарет_Показы, всего', 'Оплата за заказ_Показы, всего', 'Органика_Показы, всего', 'Вывод в топ_Показы на карточке товара', 'Трафарет_Показы на карточке товара', 'Оплата за заказ_Показы на карточке товара', 'Органика_Показы на карточке товара', 'Вывод в топ_Показы в поиске и каталоге', 'Трафарет_Показы в поиске и каталоге', 'Оплата за заказ_Показы в поиске и каталоге', 'Органика_Показы в поиске и каталоге', 'Вывод в топ_Позиция в поиске и каталог

In [63]:
df_final_db_all_features[df_final_db_all_features['Дата'] == '2025-11-17']['Показы, всего'].sum()

np.float32(175311470.0)

In [64]:
# === SQL СЦЕПКИ ОЗОН ===
sql = """
SELECT scepka.[id]
      ,scepka.[offer_id]
      ,scepka.[product_id]
      ,sku.fbo_sku as [Артикул OZ]
      ,scepka.[group_value] as [Текущая склейка]
      ,sku.[article]
      ,scepka.[updated_at] as [Дата Обновления]
  FROM [DBReport].[mp].[ozon_scepka] scepka
  JOIN [DBReport].[mp].[ozon_sku] sku 
  ON  scepka.[product_id] = sku.[product_id] 
  and sku.actual = 1
"""
df_links = pd.read_sql(sql, engine)
df_links['Артикул OZ'] = df_links['Артикул OZ'].astype(str)
df_links.to_excel(os.path.join(FOLDER_PATH, f"Склейки товаров\\OZ\\{df_links['Дата Обновления'].iloc[0].strftime('%d.%m.%Y')}_Склейка Товаров_OZ.xlsx"))

In [65]:
df_links['Текущая склейка'].unique()

array(['W8429001', '1716', '862', ..., 'W0650480', 'W2650077', None],
      shape=(29823,), dtype=object)

In [66]:
df_final_db_all_features = pd.merge(df_final_db_all_features, df_links[["Артикул OZ", "Текущая склейка"]], how='left', on='Артикул OZ')

In [67]:
df_final_db_all_features[df_final_db_all_features['Дата'] == '2025-11-17']['Расход, ₽'].sum()

np.float64(12742959.609999998)

In [68]:
df_final_db_all_features['Дата'] = pd.to_datetime(df_final_db_all_features['Дата'], format='%Y-%m-%d', errors='coerce').dt.strftime('%d.%m.%Y')

In [69]:
count = 0
for item in list(df_final_db_all_features.columns):
    print(f'{{"{item}", type {str(type(df_final_db_all_features[item].unique()[0]))}}}, ', end="")
    count +=1
    if count == 5:
        print("\n", end="")
        count = 0

{"Дата", type <class 'str'>}, {"Артикул", type <class 'str'>}, {"Артикул OZ", type <class 'str'>}, {"Наименование", type <class 'str'>}, {"Коллекция", type <class 'str'>}, 
{"Бренд", type <class 'str'>}, {"Сезон", type <class 'str'>}, {"Направление", type <class 'str'>}, {"Розничный отдел", type <class 'str'>}, {"Модель", type <class 'str'>}, 
{"Группа", type <class 'str'>}, {"Бизнес-группа", type <class 'str'>}, {"Техсегмент", type <class 'str'>}, {"Байер", type <class 'str'>}, {"Две последние коллекции", type <class 'str'>}, 
{"Основной артикул", type <class 'str'>}, {"Себестоимость с НДС", type <class 'numpy.float64'>}, {"Процент выкупа", type <class 'numpy.float64'>}, {"НДС", type <class 'numpy.float64'>}, {"Ответственный за группу", type <class 'str'>}, 
{"Группа для отчетов", type <class 'str'>}, {"ID кампании", type <class 'numpy.float64'>}, {"Инструмент", type <class 'NoneType'>}, {"Место размещения", type <class 'NoneType'>}, {"SKU из объединенной карточки", type <class 'numpy

In [70]:
print(len(list(df_final_db_all_features.columns)))

135


In [71]:
# Сохранение финальной таблицы
import pyarrow as pa
import pyarrow.csv as csv
table = pa.Table.from_pandas(df_final_db_all_features)
csv.write_csv(table, os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon_New.csv"))
# df_final_db_all_features.to_csv(os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon.csv"), index=False)

In [72]:
# del df_date_features
# gc.collect()

In [73]:
# Функция для обновления Excel-файла с циклом попыток
def update_and_save_excel(file_path, new_file_path):
    max_attempts = 10  # Максимальное количество попыток
    attempt = 0

    while attempt < max_attempts:
        attempt += 1
        print(f"Попытка {attempt} обновить файл '{os.path.basename(file_path)}'...")

        try:
            # Открываем Excel приложение
            excel = win32.Dispatch("Excel.Application")
            excel.DisplayAlerts = False  # Отключает предупреждения Excel

            try:
                # Открываем книгу
                workbook = excel.Workbooks.Open(file_path)

                # Выполняем обновление всех данных (эквивалентно "Обновить всё" в Excel)
                print("Выполняем обновление данных...")
                workbook.RefreshAll()
                excel.CalculateUntilAsyncQueriesDone()  # Дожидаемся завершения обновления

                # Сохраняем оригинальный файл в FOLDER_PATH_FOR_DB
                workbook.SaveAs(file_path)
                print(f"Файл успешно сохранен с оригинальным именем в '{os.path.dirname(file_path)}'.")

                # Сохраняем файл с новым именем в FOLDER_PATH_FEATURES
                workbook.SaveAs(new_file_path)
                print(f"Файл успешно сохранен как '{os.path.basename(new_file_path)}'.")

                return True  # Успешное завершение

            except Exception as e:
                print(f"Ошибка при обновлении или сохранении файла: {e}")
            finally:
                # Закрываем книгу и выходим из Excel
                if 'workbook' in locals():
                    workbook.Close(SaveChanges=False)
                excel.Quit()

        except Exception as e:
            print(f"Ошибка при работе с Excel: {e}")

        # Если произошла ошибка, ждем перед следующей попыткой
        if attempt < max_attempts:
            print(f"Пауза перед следующей попыткой ({attempt + 1}/{max_attempts})...")
            time.sleep(60)  # Пауза 5 секунд

    return False  # Все попытки завершились неудачно

In [ ]:
# 19. Обновить файл "Показы и затраты ОЗ_2.0.xlsx"
try:
    print("Подготовка данных для ДБ завершена.")
    # input("Начать обновление файлов ДБ? Для подтверждения нажмите Enter...")
    print("Начинаем обновлять файл 'Показы и затраты ОЗ_2.0.xlsx'...")
    start_time = time.time()  # Запускаем таймер

    # Путь к исходному файлу
    file_path_shows_expenses = os.path.join(FOLDER_PATH_FOR_DB, "Показы и затраты ОЗ_2.0_Test.xlsx")

    if os.path.exists(file_path_shows_expenses):
        # Создаем новое имя файла с текущей датой без года
        current_month_day = time.strftime("%d.%m")  # Текущая дата в формате ДД.ММ
        new_file_name = f"Показы и затраты ОЗ_2.0 {current_month_day}_Test.xlsx"
        new_file_path = os.path.join(FOLDER_PATH_FEATURES, new_file_name)

        # Путь для сохранения в дополнительную папку FOLDER_PATH_DUDL
        dudl_file_path = os.path.join(FOLDER_PATH_DUDL, new_file_name)

        # Удаляем старые файлы из FOLDER_PATH_DUDL
        try:
            if os.path.exists(FOLDER_PATH_DUDL):
                for filename in os.listdir(FOLDER_PATH_DUDL):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ОЗ_2\.0 (\d{2}\.\d{2})\_Test.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_DUDL, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_DUDL}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_DUDL}': {delete_error}")

        # Удаляем старые файлы из FOLDER_PATH_FEATURES
        try:
            if os.path.exists(FOLDER_PATH_FEATURES):
                for filename in os.listdir(FOLDER_PATH_FEATURES):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ОЗ_2\.0 (\d{2}\.\d{2})\_Test.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_FEATURES, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_FEATURES}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_FEATURES}': {delete_error}")

        # Пытаемся обновить и сохранить файл
        success = update_and_save_excel(file_path_shows_expenses, new_file_path)
        # success = True

        if not success:
            # Если все попытки неудачны, выводим сообщение пользователю
            while not success:
                input("Обновить Excel файл не получилось. Закройте все открытые файлы и нажмите любую кнопку для повторной попытки.")
                success = update_and_save_excel(file_path_shows_expenses, new_file_path)

            print("Файл успешно обновлен после повторной попытки.")

        # После успешного обновления копируем файл в папку FOLDER_PATH_DUDL
        if success:
            try:
                shutil.copy(new_file_path, dudl_file_path)
                print(f"Файл успешно скопирован в папку '{FOLDER_PATH_DUDL}'.")
            except Exception as copy_error:
                print(f"Ошибка при копировании файла в папку '{FOLDER_PATH_DUDL}': {copy_error}")

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Файл успешно обновлен и сохранен. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Показы и затраты ОЗ_2.0.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при обработке файла 'Показы и затраты ОЗ_2.0.xlsx': {e}")

Подготовка данных для ДБ завершена.
Начинаем обновлять файл 'Показы и затраты ОЗ_2.0.xlsx'...
Файл 'Показы и затраты ОЗ_2.0 01.12_Test.xlsx' удален из папки '\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ'.
Файл 'Показы и затраты ОЗ_2.0 01.12_Test.xlsx' удален из папки '\\kari.local\public\all\Analytics\Marketplaceanalytics\Дашбоард по рекламным кампаниям'.
Попытка 1 обновить файл 'Показы и затраты ОЗ_2.0_Test.xlsx'...
Выполняем обновление данных...
